<a href="https://colab.research.google.com/github/MuhammadSharie-R/portfolio/blob/main/PoliticalBias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
#Loading the NLTK Dictionaries
nltk.download ('stopwords', quiet=True)
nltk.download ('wordnet', quiet=True)
nltk.download ('omw-1.4', quiet=True)
#Loading and Preparing the Data
print("Loading dataset...")
df=pd.read_csv('bias_clean.csv')
# Drop any rows that are missing the text or the bias label
df = df.dropna(subset=['page_text', 'bias'])
#Setting up the Cleaners
lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))
#Glue the Title and the Article Body together
print("Combining headlines and article text...")
df['full_text'] = df['title'].astype(str) + " " + df['page_text'].astype(str)
#cleaning engine
def advanced_clean_text(text):
  # Convert to string and lowercase
  text = str(text).lower()
  # Remove URLs
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  # Remove punctuation and numbers (keep only letters and spaces)
  text = re.sub(r'[^a-z\s]', '', text)
  # Tokenize and Lemmatize, removing stopwords
  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
  # Stitch back together
  return " ".join(words)
# 4. Apply the function to the page_text column
print("Applying Lemmatization and cleaning text (this may take a minute or two)...")
df['cleaned_text'] = df['full_text'].apply(advanced_clean_text)
print("Text cleaning complete! Moving to next step.")

Loading dataset...
Combining headlines and article text...
Applying Lemmatization and cleaning text (this may take a minute or two)...
Text cleaning complete! Moving to next step.


In [ ]:
#convert cleaned English words into a matrix of numbers (Vectorization)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Isolate the Features (X) and the Target (y)
X = df['cleaned_text']
y = df['bias']

# 2. Split the data into Training and Testing sets
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Initialize the TF-IDF Vectorizer (The Math Engine)
print("Setting up the TF-IDF Vectorizer...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# 4. Convert the English text into Mathematical Matrices
print("Converting words to numbers. This might take a few seconds...")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Vectorization complete!")
print(f"Training data shape: {X_train_tfidf.shape}")
print(f"Testing data shape: {X_test_tfidf.shape}")

Splitting data into training and testing sets...
Setting up the TF-IDF Vectorizer...
Converting words to numbers. This might take a few seconds...
TF-IDF Vectorization complete!
Training data shape: (8665, 5000)
Testing data shape: (2167, 5000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# 1. Initialize the Models (The "Brains")
print("Initializing Machine Learning Models...")
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Support Vector Machine (SVM)": SVC(kernel='linear')
}

# 2. Setup a Dictionary to Store the Report Cards
results_dict = {'Model': [], 'Precision': [], 'Recall': [], 'F1-Score': [], 'Accuracy': []}

# 3. The Training and Testing Loop
for model_name, model in models.items():
    print(f"\n" + "="*45)
    print(f"--- Now Training: {model_name} ---")

    # Step A: Study the Training Data
    model.fit(X_train_tfidf, y_train)

    # Step B: Take the Test
    y_pred = model.predict(X_test_tfidf)

    # Step C: Grade the Test
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)

    # Step D: Save the Grades for Cell 4 (The Graph)
    results_dict['Model'].append(model_name)
    results_dict['Precision'].append(precision)
    results_dict['Recall'].append(recall)
    results_dict['F1-Score'].append(f1)
    results_dict['Accuracy'].append(accuracy)

    # Step E: Print the Detailed Report Card
    print(classification_report(y_test, y_pred, zero_division=0))

print("="*45)
print("\nAll models trained and evaluated successfully! Ready for the visualization.")

Initializing Machine Learning Models...

--- Now Training: Logistic Regression ---
               precision    recall  f1-score   support

       center       0.61      0.42      0.49       296
 leaning-left       0.53      0.71      0.60       590
leaning-right       0.83      0.59      0.69       298
         left       0.55      0.43      0.48       386
        right       0.69      0.76      0.72       597

     accuracy                           0.62      2167
    macro avg       0.64      0.58      0.60      2167
 weighted avg       0.63      0.62      0.61      2167


--- Now Training: Random Forest ---
               precision    recall  f1-score   support

       center       0.75      0.41      0.53       296
 leaning-left       0.52      0.85      0.65       590
leaning-right       0.91      0.67      0.77       298
         left       0.68      0.44      0.53       386
        right       0.77      0.76      0.77       597

     accuracy                           0.66      